In [1]:
%pip install pandas sqlalchemy pymysql


   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   -------- ------------------------------- 10.2/46.7 kB ? eta -:--:--
   -------- ------------------------------- 10.2/46.7 kB ? eta -:--:--
   -------- ------------------------------- 10.2/46.7 kB ? eta -:--:--
   -------- ------------------------------- 10.2/46.7 kB ? eta -:--:--
   -------- ---

In [5]:
import pandas as pd
from sqlalchemy import create_engine, URL
from getpass import getpass

password = getpass("Enter your MySQL password: ")

connection_url = URL.create(
    "mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    port=3307,
    database="bingeplay"
)

engine = create_engine(connection_url)

pd.read_sql("SELECT DATABASE() AS connected_database;", engine)

Enter your MySQL password:  ········


,connected_database
0,bingeplay


# BingePlay — Streaming Analytics Project
**Name:** Kalyani Pandav  
**Tools:** MySQL, Python, Pandas, SQLAlchemy  
**Analysis date:** 30 June 2024

In [6]:
query = """
SELECT
    COUNT(*) AS active_subscriptions,
    SUM(monthly_price_inr) AS monthly_revenue_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""

pd.read_sql(query, engine)

,active_subscriptions,monthly_revenue_inr
0,2340,784260.0


## Q1 — Active Revenue

**Active subscriptions:** 2,340  
**Total monthly recurring revenue:** ₹7,84,260

As of 30 June 2024, BingePlay has 2,340 active subscriptions generating ₹7,84,260 in monthly recurring revenue. Only subscriptions with active status and an end date after 30 June or no end date are included.

In [7]:
query = """
SELECT
    MONTH(signup_date) AS month,
    COUNT(*) AS signup_count
FROM users
WHERE signup_date >= '2024-01-01'
  AND signup_date < '2024-07-01'
GROUP BY MONTH(signup_date)
ORDER BY month;
"""

q2_result = pd.read_sql(query, engine)
display(q2_result)

highest = q2_result["signup_count"].max()
display(q2_result[q2_result["signup_count"] == highest])

,month,signup_count
0,1,350
1,2,400
2,3,500
3,4,550
4,5,600
5,6,600


,month,signup_count
4,5,600
5,6,600


## Q2 — Signup Momentum

| Month | Signup Count |
|---|---:|
| January | 350 |
| February | 400 |
| March | 500 |
| April | 550 |
| May | 600 |
| June | 600 |

**Highest signups:** May and June, with 600 signups each.

Monthly signups increased from 350 in January to 600 in May and remained at 600 in June. Total signups were 3,000.

In [8]:
query = """
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS completion_rate_pct
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY total_sessions DESC;
"""

pd.read_sql(query, engine)

,device_type,total_sessions,total_watch_minutes,avg_watch_minutes,completion_rate_pct
0,Mobile,50172,1504355.0,29.98,60.24
1,TV,27981,840595.0,30.04,59.98
2,Laptop,15105,453434.0,30.02,60.51
3,Tablet,7091,210733.0,29.72,59.79


## Q3 — Device Analytics

| Device | Sessions | Total Watch Minutes | Average Minutes | Completion Rate |
|---|---:|---:|---:|---:|
| Mobile | 50,172 | 1,504,355 | 29.98 | 60.24% |
| TV | 27,981 | 840,595 | 30.04 | 59.98% |
| Laptop | 15,105 | 453,434 | 30.02 | 60.51% |
| Tablet | 7,091 | 210,733 | 29.72 | 59.79% |

Mobile recorded the most sessions and total watch minutes.
Laptop had the highest completion rate at 60.51%.
The two sessions with NULL user IDs were excluded.

In [9]:
query = """
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2
    ) AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""

display(pd.read_sql(query, engine))

query_positive = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN stars IN (4, 5) THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS four_or_five_star_pct
FROM ratings;
"""

display(pd.read_sql(query_positive, engine))

,stars,rating_count,percentage
0,1,234,4.68
1,2,352,7.04
2,3,847,16.94
3,4,1781,35.62
4,5,1786,35.72


,four_or_five_star_pct
0,71.34


## Q4 — Rating Distribution

| Stars | Rating Count | Percentage |
|---|---:|---:|
| 1 | 234 | 4.68% |
| 2 | 352 | 7.04% |
| 3 | 847 | 16.94% |
| 4 | 1,781 | 35.62% |
| 5 | 1,786 | 35.72% |

**Total ratings:** 5,000  
**Ratings with 4 or 5 stars:** 71.34%

Most ratings were positive, with 3,567 out of 5,000 ratings receiving 4 or 5 stars.

In [10]:
query = """
SELECT
    CASE
        WHEN is_original = 1 THEN 'BingePlay Originals'
        ELSE 'Acquired Content'
    END AS content_type,
    COUNT(*) AS number_of_shows,
    ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
    ROUND(AVG(release_year), 2) AS avg_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""

pd.read_sql(query, engine)

,content_type,number_of_shows,avg_imdb_rating,avg_release_year
0,BingePlay Originals,30,7.92,2020.37
1,Acquired Content,70,6.63,2020.73


## Q5 — Originals vs Acquired

| Content Type | Number of Shows | Average IMDb Rating | Average Release Year |
|---|---:|---:|---:|
| BingePlay Originals | 30 | 7.92 | 2020.37 |
| Acquired Content | 70 | 6.63 | 2020.73 |

BingePlay Originals perform better on IMDb ratings, averaging 7.92 compared with 6.63 for acquired content. The difference between the rounded averages is 1.29 rating points.

In [11]:
query = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        session_date
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date >= '2024-04-01'
      AND session_date < '2024-07-01'
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
),
user_counts AS (
    SELECT user_id, COUNT(*) AS binge_day_count
    FROM binge_days
    GROUP BY user_id
)
SELECT
    user_id,
    binge_day_count,
    (SELECT COUNT(*) FROM binge_days) AS total_binge_days
FROM user_counts
WHERE binge_day_count = (
    SELECT MAX(binge_day_count) FROM user_counts
)
ORDER BY user_id;
"""

pd.read_sql(query, engine)

,user_id,binge_day_count,total_binge_days
0,U02956,8,414


## Q6 — Binge Day Detection

**Total binge days in Q2 2024:** 414  
**User with the most binge days:** U02956  
**Binge days for this user:** 8

Between 1 April and 30 June 2024, there were 414 qualifying user-show-date combinations with at least five sessions. User U02956 had the most, with 8.

In [12]:
query = """
SELECT
    COUNT(*) AS total_q1_signups,
    SUM(
        CASE WHEN NOT EXISTS (
            SELECT 1
            FROM watch_sessions ws
            WHERE ws.user_id = u.user_id
        ) THEN 1 ELSE 0 END
    ) AS never_watched_users
FROM users u
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-04-01';
"""

pd.read_sql(query, engine)

,total_q1_signups,never_watched_users
0,1250,226.0


## Q7 — Q1 Signups Who Never Watched

**Total Q1 2024 signups:** 1,250  
**Users who never watched:** 226

Of the users who signed up between January and March 2024,
226 had no watch sessions in the dataset.

NOT EXISTS checks for the absence of matching sessions and
correctly handles the NULL user IDs in watch_sessions.

In [13]:
query = """
WITH active_subscriptions AS (
    SELECT
        user_id,
        plan,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
    WHERE status = 'active'
      AND start_date <= '2024-06-30'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)
SELECT COUNT(*) AS basic_only_premium_family_users
FROM active_subscriptions a
WHERE a.rn = 1
  AND a.plan IN ('Premium', 'Family')
  AND NOT EXISTS (
      SELECT 1
      FROM watch_sessions ws
      JOIN shows s ON s.show_id = ws.show_id
      WHERE ws.user_id = a.user_id
        AND s.min_plan IN ('Premium', 'Family')
  );
"""

pd.read_sql(query, engine)

,basic_only_premium_family_users
0,204


## Q8 — Over-paying Premium/Family Users

**Answer:** 204 users.

As of 30 June 2024, 204 users with a current Premium or Family
plan had never watched a show requiring Premium or Family access.

The latest active subscription determines each user's current plan.
Users with no watch sessions are also included, as they have not
watched any higher-tier content.

In [14]:
query = """
WITH subscription_history AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date, subscription_id
        ) AS rn
    FROM subscriptions s
),
first_upgrades AS (
    SELECT
        first_sub.user_id,
        MIN(later_sub.start_date) AS first_upgrade_date
    FROM subscription_history first_sub
    JOIN subscription_history later_sub
      ON later_sub.user_id = first_sub.user_id
     AND later_sub.rn > first_sub.rn
    WHERE first_sub.rn = 1
      AND first_sub.plan = 'Basic'
      AND later_sub.plan IN ('Premium', 'Family')
      AND later_sub.start_date <= '2024-06-30'
    GROUP BY first_sub.user_id
)
SELECT
    COUNT(*) AS upgraded_active_users,
    ROUND(
        AVG(DATEDIFF(f.first_upgrade_date, u.signup_date)), 2
    ) AS avg_days_to_first_upgrade
FROM first_upgrades f
JOIN users u ON u.user_id = f.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-02-01'
  AND EXISTS (
      SELECT 1
      FROM subscriptions s
      WHERE s.user_id = u.user_id
        AND s.status = 'active'
        AND s.start_date <= '2024-06-30'
        AND (s.end_date IS NULL OR s.end_date > '2024-06-30')
  );
"""

pd.read_sql(query, engine)

,upgraded_active_users,avg_days_to_first_upgrade
0,55,64.96


## Q9 — Upgrade Success Cohort

**Number of qualifying users:** 55  
**Average days from signup to first upgrade:** 64.96 days

These users signed up in January 2024, started with Basic,
later upgraded to Premium or Family, and were still active
as of 30 June 2024.

On average, their first upgrade occurred 64.96 days after signup.

In [15]:
query = """
WITH comeback_events AS (
    SELECT DISTINCT
        a.user_id,
        a.show_id,
        a.session_date AS incomplete_date
    FROM watch_sessions a
    JOIN watch_sessions b
      ON b.user_id = a.user_id
     AND b.show_id = a.show_id
     AND b.session_date BETWEEN
         DATE_ADD(a.session_date, INTERVAL 1 DAY)
         AND DATE_ADD(a.session_date, INTERVAL 7 DAY)
    WHERE a.completed = 0
      AND a.user_id IS NOT NULL
),
show_counts AS (
    SELECT show_id, COUNT(*) AS comeback_events
    FROM comeback_events
    GROUP BY show_id
)
SELECT
    s.show_id,
    s.title,
    c.comeback_events,
    (SELECT COUNT(*) FROM comeback_events) AS total_comeback_events
FROM show_counts c
JOIN shows s ON s.show_id = c.show_id
WHERE c.comeback_events = (
    SELECT MAX(comeback_events)
    FROM show_counts
);
"""

pd.read_sql(query, engine)

,show_id,title,comeback_events,total_comeback_events
0,S088,Rayalaseema Raga,64,4345


## Q10 — Cliffhanger Comebacks

**Total comeback events:** 4,345

**Show with the most comeback events:** Rayalaseema Raga (S088), with 64 events.

A comeback occurs when a user returns to the same show
1–7 days after an incomplete session. Each distinct
user, show, and incomplete date is counted once.

In [16]:
query = """
WITH watched_weeks AS (
    SELECT DISTINCT
        user_id,
        DATE_SUB(
            session_date,
            INTERVAL WEEKDAY(session_date) DAY
        ) AS week_start
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
numbered_weeks AS (
    SELECT
        user_id,
        week_start,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY week_start
        ) AS week_number
    FROM watched_weeks
),
week_groups AS (
    SELECT
        user_id,
        CAST(
            DATEDIFF(week_start, '2000-01-03') DIV 7
            AS SIGNED
        ) - CAST(week_number AS SIGNED) AS streak_group
    FROM numbered_weeks
),
streaks AS (
    SELECT
        user_id,
        streak_group,
        COUNT(*) AS streak_weeks
    FROM week_groups
    GROUP BY user_id, streak_group
),
user_best AS (
    SELECT
        user_id,
        MAX(streak_weeks) AS longest_streak
    FROM streaks
    GROUP BY user_id
)
SELECT
    (
        SELECT COUNT(*)
        FROM user_best
        WHERE longest_streak >= 4
    ) AS users_with_4_plus_weeks,
    MAX(longest_streak) AS longest_streak_weeks,
    MIN(
        CASE
            WHEN longest_streak = (
                SELECT MAX(longest_streak)
                FROM user_best
            )
            THEN user_id
        END
    ) AS example_user_id
FROM user_best;
"""

pd.read_sql(query, engine)

,users_with_4_plus_weeks,longest_streak_weeks,example_user_id
0,1675,26,U00213


## Q11 — Consecutive Weekly Streaks

**Users with at least 4 consecutive active weeks:** 1,675

**Longest streak:** 26 weeks

**Example user with the longest streak:** U00213

An active week contains at least one watch session.
Weeks start on Monday, and multiple sessions within
the same week count as one active week.

In [18]:
query = """
WITH months AS (
    SELECT DATE('2024-05-01') AS month_start
    UNION ALL
    SELECT DATE('2024-06-01')
),
monthly_minutes AS (
    SELECT
        u.user_id,
        u.name,
        m.month_start,
        COALESCE(SUM(w.watch_minutes), 0) AS total_minutes
    FROM users u
    CROSS JOIN months m
    LEFT JOIN watch_sessions w
      ON w.user_id = u.user_id
     AND w.session_date >= m.month_start
     AND w.session_date < DATE_ADD(
         m.month_start, INTERVAL 1 MONTH
     )
    GROUP BY u.user_id, u.name, m.month_start
),
month_comparison AS (
    SELECT
        user_id,
        name,
        month_start,
        total_minutes AS june_minutes,
        LAG(total_minutes) OVER (
            PARTITION BY user_id
            ORDER BY month_start
        ) AS may_minutes
    FROM monthly_minutes
)
SELECT
    user_id,
    name,
    may_minutes,
    june_minutes,
    ROUND(
        100.0 * (may_minutes - june_minutes) / may_minutes,
        2
    ) AS drop_percentage
FROM month_comparison
WHERE month_start = '2024-06-01'
  AND may_minutes > 0
  AND june_minutes <= may_minutes * 0.5
ORDER BY drop_percentage DESC, user_id;
"""

q12_result = pd.read_sql(query, engine)
print("Total qualifying users:", len(q12_result))
display(q12_result)

Total qualifying users: 521


,user_id,name,may_minutes,june_minutes,drop_percentage
0,U00023,Amit Mukherjee,43.0,0.0,100.00
1,U00166,Shaurya Malhotra,94.0,0.0,100.00
2,U00211,Ravi Menon,336.0,0.0,100.00
3,U00225,Kritika Patil,209.0,0.0,100.00
4,U00237,Shaurya Bansal,126.0,0.0,100.00
...,...,...,...,...,...
516,U01858,Sanjay Mukherjee,296.0,147.0,50.34
517,U02530,Nandini Roy,451.0,224.0,50.33
518,U01192,Rohit Mukherjee,136.0,68.0,50.00
519,U01806,Yuvraj Raman,78.0,39.0,50.00


## Q12 — Watch-Time Drop: May to June

**Total qualifying users:** 521

These users watched for more than zero minutes in May 2024,
and their watch time fell by at least 50% in June 2024.

Users with no June sessions are counted as having zero
June watch minutes, representing a 100% drop.

For example, Amit Mukherjee (U00023) watched 43 minutes
in May and 0 minutes in June, a 100% drop.